# AETHER Quickstart: Verify the Pipeline in 2 Minutes

This notebook runs **entirely on CPU** and requires no GPU, no large datasets, and no ISRO account. It demonstrates the core Zero-DCE enhancement pipeline on a tiny sample OHRC patch so that new contributors can verify that the model architecture, inference, and visualization code all work correctly before diving into the full pipeline.

**What this notebook does:**
1. Downloads a small sample `.npy` patch (a 64×64 crop from a real Chandrayaan-2 OHRC image).
2. Loads the Zero-DCE model and runs inference.
3. Plots a **Before vs. After** comparison using the `inferno` colormap so that the extremely faint pixel variations in the PSR are visible to the human eye.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
import urllib.request
import os

device = torch.device('cpu')
print(f'Running on: {device}')

In [ ]:
# Load the tiny sample OHRC patch bundled with the repository
SAMPLE_PATH = '../data/samples/sample_patch.npy'

if not os.path.exists(SAMPLE_PATH):
    raise FileNotFoundError(f'Could not find {SAMPLE_PATH}. Make sure you are running this from the notebooks/ directory.')

# Load the patch
patch = np.load(SAMPLE_PATH)  # Expected shape: (64, 64), float32, range [0, 1]
print(f'Patch shape: {patch.shape}, dtype: {patch.dtype}')
print(f'Pixel range: [{patch.min():.4f}, {patch.max():.4f}]')
print(f'Mean brightness: {patch.mean():.4f} (expect < 0.05 for PSR patches)')

In [ ]:
class ZeroDCE(nn.Module):
    def __init__(self, channels=32, n_iter=8):
        super().__init__()
        self.n_iter = n_iter
        self.conv1 = nn.Conv2d(1, channels, 3, padding=1, padding_mode='replicate')
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, padding_mode='replicate')
        self.conv3 = nn.Conv2d(channels, channels, 3, padding=1, padding_mode='replicate')
        self.conv4 = nn.Conv2d(channels, channels, 3, padding=1, padding_mode='replicate')
        self.conv5 = nn.Conv2d(channels*2, channels, 3, padding=1, padding_mode='replicate')
        self.conv6 = nn.Conv2d(channels*2, channels, 3, padding=1, padding_mode='replicate')
        self.conv7 = nn.Conv2d(channels*2, 8, 3, padding=1, padding_mode='replicate')
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x1 = self.relu(self.conv1(x))
        x2 = self.relu(self.conv2(x1))
        x3 = self.relu(self.conv3(x2))
        x4 = self.relu(self.conv4(x3))
        x5 = self.relu(self.conv5(torch.cat([x3, x4], dim=1)))
        x6 = self.relu(self.conv6(torch.cat([x2, x5], dim=1)))
        A = torch.tanh(self.conv7(torch.cat([x1, x6], dim=1)))
        enhanced = x
        for i in range(self.n_iter):
            A_i = A[:, i:i+1, :, :]
            enhanced = enhanced + A_i * (torch.pow(enhanced, 2) - enhanced)
        return enhanced, A

model = ZeroDCE().to(device)
print(f'Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters')

# Attempt to load a checkpoint if available
CHECKPOINT_PATH = '../checkpoints/zerodce_phase2_final.pth'
if os.path.exists(CHECKPOINT_PATH):
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
    print(f'Loaded trained checkpoint from {CHECKPOINT_PATH}')
else:
    print('No checkpoint found. Running with random weights (demo mode).')
    print('To use trained weights, place zerodce_phase2_final.pth in checkpoints/')

In [ ]:
model.eval()
with torch.no_grad():
    x = torch.from_numpy(patch).unsqueeze(0).unsqueeze(0).float().to(device)
    enhanced, curve_maps = model(x)
    enhanced_np = enhanced.squeeze().cpu().numpy()

print(f'Enhanced patch range: [{enhanced_np.min():.4f}, {enhanced_np.max():.4f}]')
print(f'Enhanced mean brightness: {enhanced_np.mean():.4f}')
print(f'Brightness gain: {enhanced_np.mean() / (patch.mean() + 1e-8):.1f}x')

### Visualization
We use the `inferno` colormap because raw PSR pixel values cluster in the range [0, 0.05]. A standard grayscale plot would show nothing but black. The Min-Max stretch and `inferno` mapping reveal the faint topographic variations that the model has learned to amplify.

In [ ]:
def minmax_stretch(img):
    lo, hi = img.min(), img.max()
    if hi - lo < 1e-8:
        return np.zeros_like(img)
    return (img - lo) / (hi - lo)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1: Raw PSR patch (Min-Max stretched, Inferno)
raw_stretched = minmax_stretch(patch)
im0 = axes[0].imshow(raw_stretched, cmap='inferno', vmin=0, vmax=1)
axes[0].set_title('Raw OHRC PSR Patch\n(Min-Max Stretched)', fontsize=14, fontweight='bold')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label='Normalized Intensity')

# Panel 2: Enhanced patch (Min-Max stretched, Inferno)
enh_stretched = minmax_stretch(enhanced_np)
im1 = axes[1].imshow(enh_stretched, cmap='inferno', vmin=0, vmax=1)
axes[1].set_title('Zero-DCE Enhanced\n(Min-Max Stretched)', fontsize=14, fontweight='bold')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04, label='Normalized Intensity')

# Panel 3: Pixel intensity histograms
axes[2].hist(patch.ravel(), bins=64, alpha=0.7, label='Raw', color='steelblue', density=True)
axes[2].hist(enhanced_np.ravel(), bins=64, alpha=0.7, label='Enhanced', color='orangered', density=True)
axes[2].set_title('Pixel Intensity Distribution', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Pixel Value')
axes[2].set_ylabel('Density')
axes[2].legend(fontsize=12)
axes[2].grid(True, alpha=0.3)

plt.suptitle('AETHER: Lunar PSR Enhancement Pipeline Verification', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('quickstart_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Pipeline verification complete! If you see the inferno heatmaps above, AETHER is working.')